<img style="float: left; margin: 30px 15px 15px 15px;" src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="500" height="250" /> 
    
    
# <font color='navy'> Project: Greeks Case

<font color='black'>

- Luis Fernando Márquez Bañuelos
- Luis Eduardo Jiménez del Muro
- Fernando López Coronado
- Diego Lozoya Morales

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt

from scipy.optimize import minimize

## <font color='cornflowerblue'> Black-Scholes Class

In [2]:
class BlackScholes:
    def __init__(self, St = None, K: float = None, T: float = None, t: float = None, r: float = None, sigma: float = None):
        """
        Initialize Black-Scholes model parameters.

        Parameters:
            St (float): Spot price
            K (float): Strike price
            T (float): Time to maturity (years)
            t (float): Current time
            r (float): Risk-free interest rate
            sigma (float): Volatility of the underlying asset
        """
        if None in (St, K, T, t, r, sigma):
            self.St = 100
            self.K = 100
            self.T = 1
            self.t = 0
            self.r = 0.05
            self.sigma = 0.2
        else:
            self.St = St
            self.K = K
            self.T = T
            self.t = t
            self.r = r
            self.sigma = sigma

    # Black-Scholes formula components
    def d1(self):
        """Compute d1 in the Black-Scholes formula."""
        return (np.log(self.St / self.K) + (self.r + 0.5 * self.sigma ** 2) * (self.T - self.t)) / (self.sigma * np.sqrt(self.T - self.t))

    def d2(self):
        """Compute d2 in the Black-Scholes formula."""
        return (np.log(self.St / self.K) + (self.r - 0.5 * self.sigma ** 2) * (self.T - self.t)) / (self.sigma * np.sqrt(self.T - self.t))

    # Cumulative distribution function for a standard normal variable
    @staticmethod
    def N(x):
        """Cumulative distribution function for a standard normal variable."""
        return st.norm.cdf(x)
    
    # Option pricing methods
    def binary_cash_or_nothing_put(self):
        """Price of a Binary Cash-or-Nothing Put option."""
        return self.K * np.exp(-self.r * (self.T - self.t)) * self.N(-self.d2())

    def binary_cash_or_nothing_call(self):
        """Price of a Binary Cash-or-Nothing Call option."""
        return self.K * np.exp(-self.r * (self.T - self.t)) * self.N(self.d2())
    
    def binary_asset_or_nothing_put(self):
        """Price of a Binary Asset-or-Nothing Put option."""
        return self.St * self.N(-self.d1())

    def binary_asset_or_nothing_call(self):
        """Price of a Binary Asset-or-Nothing Call option."""
        return self.St * self.N(self.d1())
    
    # Financial options pricing methods (European options)
    def financial_call(self):
        """Price of a European Call option."""
        return self.St * self.N(self.d1()) - self.K * np.exp(-self.r * (self.T - self.t)) * self.N(self.d2())
    
    def financial_put(self):
        """Price of a European Put option."""
        return self.K * np.exp(-self.r * (self.T - self.t)) * self.N(-self.d2()) - self.St * self.N(-self.d1())
    
    # Greeks options pricing methods
    def delta_call(self):
        """Delta of a European Call option."""
        return self.N(self.d1())
    
    def delta_put(self):
        """Delta of a European Put option."""
        return self.N(self.d1()) - 1
    
    def gamma(self):
        """Gamma of the option."""
        return st.norm.pdf(self.d1()) / (self.St * self.sigma * np.sqrt(self.T - self.t))
    
    def vega(self):
        """Vega of the option."""
        return self.St * st.norm.pdf(self.d1()) * np.sqrt(self.T - self.t)
    
    def theta_call(self):
        """Theta of a European Call option."""
        return (-self.St * st.norm.pdf(self.d1()) * self.sigma / (2 * np.sqrt(self.T - self.t)) - self.r * self.K * np.exp(-self.r * (self.T - self.t)) * self.N(self.d2()))
    
    def theta_put(self):
        """Theta of a European Put option."""
        return (-self.St * st.norm.pdf(self.d1()) * self.sigma / (2 * np.sqrt(self.T - self.t)) + self.r * self.K * np.exp(-self.r * (self.T - self.t)) * self.N(-self.d2()))
    
    def rho_call(self):
        """Rho of a European Call option."""
        return self.K * (self.T - self.t) * np.exp(-self.r * (self.T - self.t)) * self.N(self.d2())
    
    def rho_put(self):
        """Rho of a European Put option."""
        return -self.K * (self.T - self.t) * np.exp(-self.r * (self.T - self.t)) * self.N(-self.d2())

    @staticmethod
    def calculate_greeks_df(df, option_type='call'):
        """
        Calculate the 5 Greeks for each row in a DataFrame.
        
        Parameters:
            df (pd.DataFrame): DataFrame with columns 'St', 'K', 'T', 't', 'r', 'sigma'
            option_type (str): Either 'call' or 'put'
        
        Returns:
            pd.DataFrame: Original DataFrame with added columns for Delta, Gamma, Vega, Theta, and Rho
        """
        if option_type.lower() not in ['call', 'put']:
            raise ValueError("option_type must be either 'call' or 'put'")
        
        # Create a copy to avoid modifying the original
        result_df = df.copy()
        
        # Initialize lists to store Greeks
        deltas = []
        gammas = []
        vegas = []
        thetas = []
        rhos = []
        
        # Calculate Greeks for each row
        for idx, row in df.iterrows():
            bs = BlackScholes(
                St=row['St'],
                K=row['K'],
                T=row['T'],
                t=row['t'],
                r=row['r'],
                sigma=row['sigma']
            )
            
            if option_type.lower() == 'call':
                deltas.append(bs.delta_call())
                thetas.append(bs.theta_call())
                rhos.append(bs.rho_call())
            else:  # put
                deltas.append(bs.delta_put())
                thetas.append(bs.theta_put())
                rhos.append(bs.rho_put())
            
            # Gamma and Vega are the same for both calls and puts
            gammas.append(bs.gamma())
            vegas.append(bs.vega())
        
        # Add Greeks as new columns
        result_df['Delta'] = deltas
        result_df['Gamma'] = gammas
        result_df['Vega'] = vegas
        result_df['Theta'] = thetas
        result_df['Rho'] = rhos
        
        return result_df

## <font color='cornflowerblue'> Data Preprocessing

In [3]:
data = pd.read_excel('options.xlsx', sheet_name='Project Data 2')
data

,Contract Name,Last Trade Date (EDT),Strike,Last Price,Bid,Ask,Change,% Change,Volume,Open Interest,Implied Volatility,Type,S0,Date,Maturity,T,Implied Vol
0,AAPL251121C00200000,2025-02-09 15:52:00,200,33.90,33.80,34.15,-2.00,-0.0557,274,2090,0.3690,Call,229.72,2025-09-02,2025-11-21,0.219178,0.307079
1,AAPL251121C00210000,2025-02-09 15:03:00,210,24.85,25.30,25.90,-3.15,-0.1125,306,2036,0.3381,Call,229.72,2025-09-02,2025-11-21,0.219178,0.265185
2,AAPL251121C00215000,2025-02-09 15:53:00,215,21.60,21.45,21.85,-1.99,-0.0844,155,3156,0.3183,Call,229.72,2025-09-02,2025-11-21,0.219178,0.276876
3,AAPL251121C00220000,2025-02-09 15:59:00,220,17.95,17.90,18.25,-1.53,-0.0785,241,7527,0.3052,Call,229.72,2025-09-02,2025-11-21,0.219178,0.267559
4,AAPL251121C00225000,2025-02-09 15:48:00,225,14.75,14.60,14.85,-1.25,-0.0781,604,6117,0.2909,Call,229.72,2025-09-02,2025-11-21,0.219178,0.262453
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,AAPL251121C00235000,10/30/2025 3:59 PM,235,37.05,35.70,38.80,2.68,0.0780,280,14757,0.5430,Call,271.40,2025-11-20,2025-11-21,0.002740,0.272038
146,AAPL251121C00240000,10/30/2025 3:58 PM,240,32.40,31.00,33.90,1.41,0.0455,519,14112,0.4933,Call,271.40,2025-11-20,2025-11-21,0.002740,0.312957
147,AAPL251121C00250000,10/30/2025 3:58 PM,250,23.10,21.75,24.00,1.28,0.0587,803,21693,0.3840,Call,271.40,2025-11-20,2025-11-21,0.002740,0.292546
148,AAPL251121C00260000,10/30/2025 3:59 PM,260,14.75,13.70,15.00,1.15,0.0846,4131,27124,0.3093,Call,271.40,2025-11-20,2025-11-21,0.002740,0.276864


In [4]:
# Drop unnecessary columns
data.drop(columns=['Contract Name','Last Trade Date (EDT)','Bid','Ask','Change','% Change','Volume','Open Interest','Implied Volatility','Type'], inplace=True)

# Merge risk-free rate data
data['r'] = 0.0382
data['t'] = 0

# Rename columns for consistency
data = data.rename(columns={'Strike': 'K', 'S0': 'St', 'Implied Vol': 'sigma', 'Rf': 'r'})

# Correct r to continuous compounding
data['r'] = np.log(1 + data['r'])

data

,K,Last Price,St,Date,Maturity,T,sigma,r,t
0,200,33.90,229.72,2025-09-02,2025-11-21,0.219178,0.307079,0.037488,0
1,210,24.85,229.72,2025-09-02,2025-11-21,0.219178,0.265185,0.037488,0
2,215,21.60,229.72,2025-09-02,2025-11-21,0.219178,0.276876,0.037488,0
3,220,17.95,229.72,2025-09-02,2025-11-21,0.219178,0.267559,0.037488,0
4,225,14.75,229.72,2025-09-02,2025-11-21,0.219178,0.262453,0.037488,0
...,...,...,...,...,...,...,...,...,...
145,235,37.05,271.40,2025-11-20,2025-11-21,0.002740,0.272038,0.037488,0
146,240,32.40,271.40,2025-11-20,2025-11-21,0.002740,0.312957,0.037488,0
147,250,23.10,271.40,2025-11-20,2025-11-21,0.002740,0.292546,0.037488,0
148,260,14.75,271.40,2025-11-20,2025-11-21,0.002740,0.276864,0.037488,0


In [5]:
# Calculating proper implied volatilities with the adjusted risk-free rate
all_S0 = data['St'].values
K = data['K'].values
T = data['T'].values
t = data['t'].values
r = data['r'].values
sigma = data['sigma'].values
CallMKT = data['Last Price'].values

results = []
for i in range(len(data)):
    bounds = [[None, None]]
    apply_constraints1 = lambda x: BlackScholes(all_S0[i], K[i], T[i], t[i], r[i], x[0]).financial_call() - CallMKT[i]
    my_constraints = {'type': 'eq', 'fun': apply_constraints1}
    result = minimize(lambda x: 0.0,
                x0=np.array([sigma[i]]),
                bounds=bounds,
                constraints=my_constraints,
                method='SLSQP')
    implied_volatility = result.x[0]
    results.append(implied_volatility)

data['sigma'] = results

In [6]:
# Calculating greeks
bs = BlackScholes()
data = bs.calculate_greeks_df(data, option_type='call')
data

,K,Last Price,St,Date,Maturity,T,sigma,r,t,Delta,Gamma,Vega,Theta,Rho
0,200,33.90,229.72,2025-09-02,2025-11-21,0.219178,0.310887,0.037488,0,0.860177,6.651295e-03,2.391683e+01,-23.098943,35.879409
1,210,24.85,229.72,2025-09-02,2025-11-21,0.219178,0.268119,0.037488,0,0.800457,9.695650e-03,3.006761e+01,-24.352570,34.856081
2,215,21.60,229.72,2025-09-02,2025-11-21,0.219178,0.279207,0.037488,0,0.737234,1.086105e-02,3.507455e+01,-27.879579,32.385163
3,220,17.95,229.72,2025-09-02,2025-11-21,0.219178,0.269552,0.037488,0,0.681109,1.231796e-02,3.840391e+01,-28.807833,30.359320
4,225,14.75,229.72,2025-09-02,2025-11-21,0.219178,0.264159,0.037488,0,0.616441,1.344013e-02,4.106414e+01,-29.501485,27.804686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,235,37.05,271.40,2025-11-20,2025-11-21,0.002740,0.272038,0.037488,0,1.000000,5.496127e-24,3.017260e-22,-8.808880,0.643769
146,240,32.40,271.40,2025-11-20,2025-11-21,0.002740,1.752939,0.037488,0,0.917288,6.122139e-03,2.165692e+00,-700.947745,0.593293
147,250,23.10,271.40,2025-11-20,2025-11-21,0.002740,1.537583,0.037488,0,0.855892,1.039153e-02,3.224372e+00,-912.629952,0.573121
148,260,14.75,271.40,2025-11-20,2025-11-21,0.002740,1.393856,0.037488,0,0.734364,1.656214e-02,4.658662e+00,-1191.982944,0.505634


In [7]:
# The table is sent to an excel file, then that excel is used in the hedging strategy
#data.to_excel('op_2.xlsx', index=False)